## Clean the data

In [1]:
from src.get_data import get_player_stats, compute_expected_metric, get_lineups, parse_player_ids, compute_weighted_ts, compute_expected_net
import pandas as pd
import time

In [2]:
SEASONS       = ['2025-26']          # add more e.g. ['2022-23', '2023-24', '2024-25', '2025-26']
GROUP_SIZES   = [2, 3, 4, 5]
MIN_THRESHOLDS = {2: 100, 3: 75, 4: 50, 5: 30}   # minimum shared minutes to include a lineup

In [ ]:
all_lineups = []
all_players = []

for season in SEASONS:
    print(f"\n{'='*50}")
    print(f"Season: {season}")
    print(f"{'='*50}")

    # 1. Individual player stats
    player_df = get_player_stats(season)
    all_players.append(player_df)
    time.sleep(1)

    # 2. Build lookup dicts: player_id → metric
    player_net_lookup = dict(zip(player_df['PLAYER_ID'], player_df['NET_RATING']))
    player_off_lookup = dict(zip(player_df['PLAYER_ID'], player_df['OFF_RATING']))
    player_def_lookup = dict(zip(player_df['PLAYER_ID'], player_df['DEF_RATING']))
    player_pie_lookup = dict(zip(player_df['PLAYER_ID'], player_df['PIE']))
    player_ts_lookup  = dict(zip(player_df['PLAYER_ID'], player_df['TS_PCT']))
    player_usg_lookup = dict(zip(player_df['PLAYER_ID'], player_df['USG_PCT']))

    # 3. Lineup data for each group size
    for group_size in GROUP_SIZES:
        df = get_lineups(season, group_size)

        # Parse player IDs
        df['player_ids'] = df['GROUP_ID'].apply(parse_player_ids)

        # Apply minimum minutes filter
        min_thresh = MIN_THRESHOLDS[group_size]
        df = df[df['MIN'] >= min_thresh].copy()
        print(f"    {len(df)} lineups after {min_thresh}min filter")

        # Compute expected values from individual stats
        df['expected_net_rating'] = df['player_ids'].apply(
            lambda ids: compute_expected_net(ids, player_net_lookup)
        )
        df['expected_off_rating'] = df['player_ids'].apply(
            lambda ids: compute_expected_metric(ids, player_off_lookup)
        )
        df['expected_def_rating'] = df['player_ids'].apply(
            lambda ids: compute_expected_metric(ids, player_def_lookup)
        )
        df['expected_pie'] = df['player_ids'].apply(
            lambda ids: compute_expected_metric(ids, player_pie_lookup)
        )
        df['expected_ts_usg_weighted'] = df['player_ids'].apply(
            lambda ids: compute_weighted_ts(ids, player_ts_lookup, player_usg_lookup)
        )

        # Compute synergy deltas
        df['synergy_delta']     = df['NET_RATING'] - df['expected_net_rating']
        df['off_synergy_delta'] = df['OFF_RATING'] - df['expected_off_rating']
        df['def_synergy_delta'] = df['DEF_RATING'] - df['expected_def_rating']   # lower is better
        df['pie_synergy_delta'] = df['PIE']        - df['expected_pie']

        # Convert player_ids list → string for CSV export
        df['player_ids_str'] = df['player_ids'].apply(lambda x: ','.join(x))
        df = df.drop(columns=['player_ids'])

        all_lineups.append(df)


Season: 2025-26
  Fetching individual player stats for 2025-26...
  Fetching 2-man lineups for 2025-26...


In [ ]:
player_df.to_csv('player_stats.csv', index=False)

In [ ]:
df_combined = pd.concat(all_lineups[0:4], ignore_index=True)

In [ ]:
df_combined.sort_values('NET_RATING', ascending=False)[['GROUP_ID', 'NET_RATING', 'GROUP_NAME', 'MIN', 'NET_RATING_RANK', 'MIN_RANK', 'W', 'L']].head(10)

,GROUP_ID,NET_RATING,GROUP_NAME,MIN,NET_RATING_RANK,MIN_RANK,W,L
7837,-1627936-1628983-1630198-1631119-1642349-,66.8,A. Caruso - S. Gilgeous-Alexander - I. Joe - J...,35.0,120,8,7,1
6164,-201142-1629006-1641708-1642263-,56.5,K. Durant - J. Okogie - A. Thompson - R. Sheppard,51.0,70,70,8,6
7064,-1630577-1641705-1642264-1642844-,56.4,J. Champagnie - V. Wembanyama - S. Castle - D....,63.0,85,68,22,5
7523,-1628401-1629014-1629674-1630573-1642864-,52.8,D. White - A. Simons - N. Queta - S. Hauser - ...,47.0,89,7,9,3
7123,-1627936-1628983-1630198-1642349-,50.3,A. Caruso - S. Gilgeous-Alexander - I. Joe - A...,62.0,183,42,13,1
5703,-1628401-1629674-1630573-1642864-,48.8,D. White - N. Queta - S. Hauser - H. González,64.0,108,52,16,7
7540,-1628378-1628386-1629636-1630596-1642281-,47.6,D. Mitchell - J. Allen - D. Garland - E. Moble...,35.0,145,9,4,1
7032,-1629640-1630170-1641705-1642844-,47.0,K. Johnson - D. Vassell - V. Wembanyama - D. H...,95.0,95,35,21,2
7106,-1627936-1628983-1631119-1642349-,46.5,A. Caruso - S. Gilgeous-Alexander - J. William...,80.0,206,24,12,2
5749,-1628378-1629731-1630241-1642281-,46.3,D. Mitchell - D. Wade - S. Merrill - J. Tyson,80.0,173,26,7,4


### We need to remake all the rank columns from the dataset, they are not properly calculated

In [ ]:
# 1. Define columns where Higher values get a better Rank (Rank 1 = Highest)
high_is_better = [
    'GP', 'W', 'W_PCT', 'MIN', 'E_OFF_RATING', 'OFF_RATING', 'E_NET_RATING', 
    'NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 
    'REB_PCT', 'EFG_PCT', 'TS_PCT', 'PIE', 'SUM_TIME_PLAYED', 'synergy_delta',
    'off_synergy_delta', 'pie_synergy_delta'
]

# 2. Define columns where Lower values get a better Rank (Rank 1 = Lowest)
low_is_better = [
    'L', 'E_DEF_RATING', 'DEF_RATING', 'TM_TOV_PCT', 'def_synergy_delta'
]

# Re-calculate ranks for 'High is Better'
for col in high_is_better:
    rank_col = f"{col}_RANK"
    # ascending=False means the largest value gets Rank 1
    df_combined[rank_col] = df_combined[col].rank(ascending=False, method='min')

# Re-calculate ranks for 'Low is Better'
for col in low_is_better:
    rank_col = f"{col}_RANK"
    # ascending=True means the smallest value gets Rank 1
    df_combined[rank_col] = df_combined[col].rank(ascending=True, method='min')

# Special Case: Pace (Usually ranked by speed, but can be subjective. 
# Here we'll rank fastest as #1)
pace_cols = ['E_PACE', 'PACE', 'PACE_PER40']
for col in pace_cols:
    df_combined[f"{col}_RANK"] = df_combined[col].rank(ascending=False, method='min')

In [ ]:
df_combined.sort_values('NET_RATING', ascending=False)[['GROUP_ID', 'NET_RATING', 'GROUP_NAME', 'MIN', 'NET_RATING_RANK', 'MIN_RANK', 'W', 'L']].head(10)

,GROUP_ID,NET_RATING,GROUP_NAME,MIN,NET_RATING_RANK,MIN_RANK,W,L
7837,-1627936-1628983-1630198-1631119-1642349-,66.8,A. Caruso - S. Gilgeous-Alexander - I. Joe - J...,35.0,1.0,7824.0,7,1
6164,-201142-1629006-1641708-1642263-,56.5,K. Durant - J. Okogie - A. Thompson - R. Sheppard,51.0,2.0,7574.0,8,6
7064,-1630577-1641705-1642264-1642844-,56.4,J. Champagnie - V. Wembanyama - S. Castle - D....,63.0,3.0,7076.0,22,5
7523,-1628401-1629014-1629674-1630573-1642864-,52.8,D. White - A. Simons - N. Queta - S. Hauser - ...,47.0,4.0,7699.0,9,3
7123,-1627936-1628983-1630198-1642349-,50.3,A. Caruso - S. Gilgeous-Alexander - I. Joe - A...,62.0,5.0,7119.0,13,1
5703,-1628401-1629674-1630573-1642864-,48.8,D. White - N. Queta - S. Hauser - H. González,64.0,6.0,7040.0,16,7
7540,-1628378-1628386-1629636-1630596-1642281-,47.6,D. Mitchell - J. Allen - D. Garland - E. Moble...,35.0,7.0,7824.0,4,1
7032,-1629640-1630170-1641705-1642844-,47.0,K. Johnson - D. Vassell - V. Wembanyama - D. H...,95.0,8.0,5661.0,21,2
7106,-1627936-1628983-1631119-1642349-,46.5,A. Caruso - S. Gilgeous-Alexander - J. William...,80.0,9.0,6427.0,12,2
5749,-1628378-1629731-1630241-1642281-,46.3,D. Mitchell - D. Wade - S. Merrill - J. Tyson,80.0,10.0,6427.0,7,4


In [ ]:
df_combined.to_csv('lineups_with_synergy.csv', index=False)

### Create clusters

In [ ]:
from k_means.cluster_creation import load_data, cluster_players, plot_clusters, build_archetype_lookup, enrich_lineups, archetype_compatibility_matrix, best_lineups_per_team
import json

In [ ]:
players_df, lineups_df = load_data()
players_with_archetypes, scaler, km = cluster_players(players_df)

Loading data...
  546 player-season rows
  7920 lineup rows

  440 players with 15+ games (used for clustering)

── Cluster Centroids (original scale) ──────────────────────────────
         USG_PCT  AST_PCT  OREB_PCT  DREB_PCT  TS_PCT    PIE  REB_PCT
cluster                                                              
0          0.156    0.103     0.029     0.090   0.572  0.070    0.060
1          0.145    0.088     0.120     0.191   0.634  0.109    0.155
2          0.174    0.217     0.027     0.102   0.487  0.072    0.064
3          0.267    0.225     0.066     0.213   0.593  0.151    0.140
4          0.179    0.120     0.053     0.146   0.598  0.101    0.100
5          0.131    0.083     0.063     0.136   0.500  0.061    0.099
6          0.246    0.251     0.025     0.105   0.582  0.120    0.065

── Top players per cluster (by PIE) ───────────────────────────────
  [0] 3-and-D Wing             : Jamaree Bouyea, Mikal Bridges, Zach LaVine, Isaiah Joe, Jerami Grant
  [1] Rim-Runner 

In [ ]:
plot_clusters(players_with_archetypes)

  Saved player_archetypes_pca.png


In [ ]:
archetype_lookup = build_archetype_lookup(players_with_archetypes)

In [ ]:
lineups_enriched = enrich_lineups(lineups_df, archetype_lookup)
n_matched = lineups_enriched['archetype_fingerprint'].notna().sum()

In [ ]:
compatibility = archetype_compatibility_matrix(lineups_enriched)


── Archetype Compatibility Matrix (avg synergy_delta for 2-man lineups) ──
                      3-and-D Wing  Rim-Runner  Secondary Creator  Star Engine  Versatile Big  Glue / Role Player  Primary Ball-Handler
3-and-D Wing                 -1.41        1.05              -0.49         0.94          -0.23               -0.40                 -0.13
Rim-Runner                    1.05         NaN              -1.33         0.11          -0.39                1.01                  0.31
Secondary Creator            -0.49       -1.33              -2.25        -0.73          -1.59                0.67                 -0.50
Star Engine                   0.94        0.11              -0.73          NaN           2.33               -0.82                  1.28
Versatile Big                -0.23       -0.39              -1.59         2.33           0.23               -2.02                  0.15
Glue / Role Player           -0.40        1.01               0.67        -0.82          -2.02               

In [ ]:
best_lineups_per_team(lineups_enriched, group_size=5)
# best_lineups_per_team(lineups_enriched, group_size=2)


── Top 3 lineups (group_size=5) per team by synergy_delta ──
  Saved best_lineups_per_team.csv (0 rows)


""


In [ ]:
ARCHETYPE_LABELS = {
    0: "Primary Ball-Handler",    # high USG, high AST
    1: "3-and-D Wing",            # low USG, high TS, moderate DEF value
    2: "Stretch Big",             # high TS, moderate REB, perimeter capable
    3: "Rim-Runner",              # high OREB/DREB, low USG, paint-only
    4: "Secondary Creator",       # moderate USG, moderate AST, shooting guard type
    5: "Two-Way Wing",            # balanced across everything, high PIE
    6: "Glue Guy",                # low across the board but high team impact
}


In [ ]:
fives = lineups_enriched[
        (lineups_enriched['group_size'] == 5) &
        (lineups_enriched['archetype_fingerprint_str'].notna())]

combo_summary = (
    fives.groupby('archetype_fingerprint_str')['synergy_delta']
            .agg(mean='mean', std='std', count='count')
            .query('count >= 3')
            .sort_values('mean', ascending=False)
            .head(10))

# Decode fingerprint to labels for readability
def decode(fp_str):
    try:
        ids = json.loads(fp_str.replace('(', '[').replace(')', ']'))
        return ' + '.join(ARCHETYPE_LABELS.get(i, str(i)) for i in ids)
    except Exception:
        return fp_str

combo_summary['archetype_names'] = combo_summary.index.map(decode)
print(combo_summary[['archetype_names', 'mean', 'std', 'count']].round(3).to_string())

                                                                                                 archetype_names    mean     std  count
archetype_fingerprint_str                                                                                                              
(0, 0, 1, 2, 3)            Primary Ball-Handler + Primary Ball-Handler + 3-and-D Wing + Stretch Big + Rim-Runner  28.713  26.777      3
(0, 1, 2, 3, 6)                        Primary Ball-Handler + 3-and-D Wing + Stretch Big + Rim-Runner + Glue Guy  25.600   6.971      3
(2, 2, 3, 4, 6)                            Stretch Big + Stretch Big + Rim-Runner + Secondary Creator + Glue Guy  24.307   7.025      3
(1, 3, 5, 6, 6)                                   3-and-D Wing + Rim-Runner + Two-Way Wing + Glue Guy + Glue Guy  21.180  16.789      3
(0, 0, 4, 6, 6)            Primary Ball-Handler + Primary Ball-Handler + Secondary Creator + Glue Guy + Glue Guy  21.067  27.755      3
(0, 1, 5, 6, 6)                         Primary 